In [1]:
import requests
import pandas as pd
import geopandas as gpd
import time
import datetime
import folium
from folium.plugins import MarkerCluster
import numpy as np

In [2]:
# We need to access the API and to do that, will use the map key that permits access.
MAP_KEY = '54684dde74a099b139ddbbef0f621891'

# Now let's check how many results we have

url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  test_response = requests.get(url)
  test_data = response.json()
  test_df = pd.Series(data)
  display(test_df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)

There is an issue with the query. 
Try in your browser: https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=54684dde74a099b139ddbbef0f621891


In [3]:
# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
sensor_data = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
test_df = pd.read_csv(sensor_data)
display(test_df)

,data_id,min_date,max_date
0,MODIS_NRT,2026-03-01,2026-05-20
1,MODIS_SP,2000-11-01,2026-02-28
2,VIIRS_NOAA20_NRT,2026-04-01,2026-05-20
3,VIIRS_NOAA20_SP,2018-04-01,2026-03-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-20
5,VIIRS_SNPP_NRT,2026-04-01,2026-05-19
6,VIIRS_SNPP_SP,2012-01-20,2026-03-31
7,LANDSAT_NRT,2022-06-20,2026-05-19
8,GOES_NRT,2022-08-09,2026-05-20
9,BA_MODIS,2000-11-01,2026-02-01


In [4]:
# We are particularly interested in the wildfires in Australia and so will select this information using a bounding box. Aus = 110 -55, 180 -10
modis_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_NRT/110,-50,160,-11/3'
modis_nrt_df = pd.read_csv(modis_nrt_url)

modis_sp_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_SP/110,-50,160,-11/3'
modis_sp_df = pd.read_csv(modis_sp_url)

viirs_noaa20_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/110,-50,160,-11/3'
viirs_noaa20_nrt_df = pd.read_csv(viirs_noaa20_nrt_url)

viirs_noaa20_sp_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_SP/110,-50,160,-11/3'
viirs_noaa20_sp_df = pd.read_csv(viirs_noaa20_sp_url)

viirs_noaa21_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA21_NRT/110,-50,160,-11/3'
viirs_noaa21_nrt_df = pd.read_csv(viirs_noaa21_nrt_url)

viirs_snpp_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_NRT/110,-50,160,-11/3'
viirs_snpp_nrt_df = pd.read_csv(viirs_snpp_nrt_url)

viirs_snpp_sp_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_SP/110,-50,160,-11/3'
viirs_snpp_sp_df = pd.read_csv(viirs_snpp_sp_url)

In [5]:
# Create a funtion that checks time of aquisition and calculates time delta
def add_time_since_acq(df):
    df["acq_date"] = pd.to_datetime(df["acq_date"])   # Ensuring date is in datetime format
    df["acq_datetime"] = pd.to_datetime(
        df["acq_date"].astype(str) + df["acq_time"].astype(str).str.zfill(4),    # Turns all values into 4 digit HHHH format
        format = "%Y-%m-%d%H%M"
        ).dt.tz_localize('UTC')

    current_time = pd.Timestamp.now('UTC')
    df["time_since_detection"] = current_time - df["acq_datetime"]

    # Calculate hours since detection
    df["hours_since"] = df["time_since_detection"].dt.total_seconds()/3600

    df["acq_date_str"] = df["time_since_detection"].astype(str)
    df["time_since_detection_str"] = df["acq_date"].astype(str)
    df["acq_datetime_str"] = df["acq_datetime"].astype(str)


    df = df.drop(columns=["time_since_detection", "acq_date", "acq_datetime"])


    return df


modis_nrt_df = add_time_since_acq(modis_nrt_df)
print(modis_nrt_df.iloc[100,:])
modis_nrt_df.dtypes

latitude                                    -13.40639
longitude                                   131.61371
brightness                                     312.19
scan                                             1.39
track                                            1.17
acq_time                                          629
satellite                                        Aqua
instrument                                      MODIS
confidence                                         44
version                                        6.1NRT
bright_t31                                     292.09
frp                                              9.81
daynight                                            D
hours_since                                 51.201075
acq_date_str                   2 days 03:12:03.869024
time_since_detection_str                   2026-05-18
acq_datetime_str            2026-05-18 06:29:00+00:00
Name: 100, dtype: object


latitude                    float64
longitude                   float64
brightness                  float64
scan                        float64
track                       float64
acq_time                      int64
satellite                       str
instrument                      str
confidence                    int64
version                         str
bright_t31                  float64
frp                         float64
daynight                        str
hours_since                 float64
acq_date_str                    str
time_since_detection_str        str
acq_datetime_str                str
dtype: object

In [6]:
# Convert to GeoDataFrame (WGS84)
modis_nrt_gdf = gpd.GeoDataFrame(
    modis_nrt_df, 
    geometry=gpd.points_from_xy(
        modis_nrt_df["longitude"], 
        modis_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_noaa20_nrt_gdf = gpd.GeoDataFrame(
    viirs_noaa20_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_noaa20_nrt_df["longitude"], 
        viirs_noaa20_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_noaa21_nrt_gdf = gpd.GeoDataFrame(
    viirs_noaa21_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_noaa21_nrt_df["longitude"], 
        viirs_noaa21_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_snpp_nrt_gdf = gpd.GeoDataFrame(
    viirs_snpp_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_snpp_nrt_df["longitude"], 
        viirs_snpp_nrt_df["latitude"]
    ),
    crs="EPSG:4326")


print(f"Found {len(modis_nrt_gdf)} fire records.")
print(f"Found {len(viirs_noaa20_nrt_gdf)} fire records.")
print(f"Found {len(viirs_noaa21_nrt_gdf)} fire records.")
print(f"Found {len(viirs_snpp_nrt_gdf)} fire records.")

Found 490 fire records.
Found 2310 fire records.
Found 2862 fire records.
Found 1602 fire records.


In [7]:
modis_nrt_gdf = gpd.GeoDataFrame(
    modis_nrt_df, 
    geometry=gpd.points_from_xy(
        modis_nrt_df["longitude"], 
        modis_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

# Initialise a map centered on Australia
australia_map = folium.Map(
    location=[-28.281828, 136.145401],
    zoom_start=5,
    tiles="CyclOSM",  # A clean, light basemap
)

modis_nrt_gdf.explore(
    m=australia_map, 
    column='confidence', # This is what the scale is based on.
    name='MODIS NRT',  
    tooltip=['brightness', 'confidence'], 
    cmap='YlOrRd',
    style_kwds={'fillOpacity': 0.8, 'color': 'white', 'weight': 0.1},
    show=True
)

In [10]:
aus_lgas = gpd.read_file("data/ASGS_Ed3_Non_ABS_Structures_GDA2020_updated_2025/ASGS_Ed3_Non_ABS_Structures_GDA2020_updated_2025.gpkg",
                        layer="LGA_2025_AUST_GDA2020"
                        ).to_crs(epsg=4326)

# 1. Spatial joins
modis_nrt_joined = gpd.sjoin(modis_nrt_gdf, aus_lgas, how="inner", predicate="within")
viirs_noaa20_nrt_joined = gpd.sjoin(viirs_noaa20_nrt_gdf, aus_lgas, how="inner", predicate="within")
viirs_noaa21_nrt_joined = gpd.sjoin(viirs_noaa21_nrt_gdf, aus_lgas, how="inner", predicate="within")
viirs_snpp_nrt_joined = gpd.sjoin(viirs_snpp_nrt_gdf, aus_lgas, how="inner", predicate="within")

# 2. Detect correct join key
name_col = "LGA_NAME_2025_right" if "LGA_NAME_2025_right" in modis_nrt_joined.columns else "LGA_NAME_2025"

# 3. Count fires per LGA for each dataset
modis_counts = modis_nrt_joined.groupby(name_col).size().reset_index(name="modis_count")
noaa20_counts = viirs_noaa20_nrt_joined.groupby(name_col).size().reset_index(name="noaa20_count")
noaa21_counts = viirs_noaa21_nrt_joined.groupby(name_col).size().reset_index(name="noaa21_count")
snpp_counts = viirs_snpp_nrt_joined.groupby(name_col).size().reset_index(name="snpp_count")

# 4. Rename join key for all count tables
for df in [modis_counts, noaa20_counts, noaa21_counts, snpp_counts]:
    df.rename(columns={name_col: "LGA_NAME_2025"}, inplace=True)

# 5. Merge all counts into aus_lgas
aus_lgas = (
    aus_lgas
    .merge(modis_counts, on="LGA_NAME_2025", how="left")
    .merge(noaa20_counts, on="LGA_NAME_2025", how="left")
    .merge(noaa21_counts, on="LGA_NAME_2025", how="left")
    .merge(snpp_counts, on="LGA_NAME_2025", how="left")
)

# Replace NaN with 0
aus_lgas[["modis_count", "noaa20_count", "noaa21_count", "snpp_count"]] = \
    aus_lgas[["modis_count", "noaa20_count", "noaa21_count", "snpp_count"]].fillna(0)

# 6. Total fire count
aus_lgas["fire_count"] = aus_lgas[
    ["modis_count", "noaa20_count", "noaa21_count", "snpp_count"]
].sum(axis=1)

# 7. Fire density
aus_lgas["fire_density"] = (aus_lgas["fire_count"] / aus_lgas["AREA_ALBERS_SQKM"]).round(2)

aus_lgas.head(40)


,LGA_CODE_2025,LGA_NAME_2025,STATE_CODE_2021,STATE_NAME_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,geometry,modis_count,noaa20_count,noaa21_count,snpp_count,fire_count,fire_density
0,10050,Albury,1,New South Wales,AUS,Australia,305.6386,"MULTIPOLYGON (((146.86566 -36.07292, 146.8663 ...",0.0,0.0,0.0,0.0,0.0,0.0
1,10180,Armidale,1,New South Wales,AUS,Australia,7809.4406,"MULTIPOLYGON (((152.38816 -30.52639, 152.38744...",0.0,1.0,0.0,0.0,1.0,0.0
2,10250,Ballina,1,New South Wales,AUS,Australia,484.9692,"MULTIPOLYGON (((153.57106 -28.87381, 153.57106...",0.0,0.0,0.0,0.0,0.0,0.0
3,10300,Balranald,1,New South Wales,AUS,Australia,21690.7493,"MULTIPOLYGON (((143.00433 -33.78164, 142.99952...",0.0,0.0,0.0,0.0,0.0,0.0
4,10470,Bathurst,1,New South Wales,AUS,Australia,3817.8645,"MULTIPOLYGON (((149.84877 -33.52784, 149.84894...",0.0,0.0,0.0,0.0,0.0,0.0
5,10500,Bayside (NSW),1,New South Wales,AUS,Australia,50.6204,"MULTIPOLYGON (((151.14805 -33.92725, 151.14792...",0.0,0.0,0.0,0.0,0.0,0.0
6,10550,Bega Valley,1,New South Wales,AUS,Australia,6278.5013,"MULTIPOLYGON (((150.05261 -37.26253, 150.05267...",0.0,0.0,0.0,0.0,0.0,0.0
7,10600,Bellingen,1,New South Wales,AUS,Australia,1600.4338,"MULTIPOLYGON (((152.47659 -30.3968, 152.47321 ...",0.0,0.0,0.0,0.0,0.0,0.0
8,10650,Berrigan,1,New South Wales,AUS,Australia,2065.8878,"MULTIPOLYGON (((145.46159 -35.66977, 145.46154...",0.0,1.0,3.0,0.0,4.0,0.0
9,10750,Blacktown,1,New South Wales,AUS,Australia,238.8471,"MULTIPOLYGON (((150.87089 -33.82385, 150.87101...",0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:

# 2. Spatial Join: Count fire detections within each LGA
modis_nrt_joined = gpd.sjoin(
    modis_nrt_gdf, 
    aus_lgas, 
    how="inner", 
    predicate="within"
)

viirs_noaa20_nrt_joined = gpd.sjoin(
    viirs_noaa20_nrt_gdf, 
    aus_lgas, 
    how="inner", 
    predicate="within"
)

viirs_noaa21_nrt_joined = gpd.sjoin(
    viirs_noaa21_nrt_gdf, 
    aus_lgas, 
    how="inner", 
    predicate="within"
)

viirs_snpp_nrt_joined = gpd.sjoin(
    viirs_snpp_nrt_gdf, 
    aus_lgas, 
    how="inner", 
    predicate="within"
)


name_col = "LGA_NAME_2025_right" if "LGA_NAME_2025_right" in modis_nrt_joined.columns else "LGA_NAME_2025"

# Group by the lga name and count the occurrences
modis_counts = modis_nrt_joined.groupby(name_col).size().reset_index(name="modis_count")
noaa20_counts = viirs_noaa20_nrt_joined.groupby(name_col).size().reset_index(name="noaa20_count")
noaa21_counts = viirs_noaa21_nrt_joined.groupby(name_col).size().reset_index(name="noaaa21_count")
snpp_counts = viirs_snpp_nrt_joined.groupby(name_col).size().reset_index(name="snpp_count")

# Rename join key to match
modis_counts.rename(columns={name_col: "LGA_NAME_2025"}, inplace=True)


modis_counts = modis_counts[["LGA_NAME_2025", "modis_count"]]



# Merge the counts back into the main quarters GeoDataFrame
aus_lgas = (aus_lgas
    .merge(modis_counts, on="LGA_NAME_2025", how="left")
    .merge(noaa20_counts, on="LGA_NAME_2025", how="left")
    .merge(noaa21_counts, on="LGA_NAME_2025", how="left")
    .merge(snpp_counts, on="LGA_NAME_2025", how="left")
           )

#.fillna({"modis_count": 0})

# Calculate the density (fire detections per square kilometer)
aus_lgas["modis_count", "noaa20_count", "noaaa21_count", "snpp_count"].sum("fire_count")
aus_lgas["fire_density"] = (aus_lgas["modis_count"] / aus_lgas["AREA_ALBERS_SQKM"]).round(2)

#aus_lgas.head()